# RAG: 검색증강생성 완전 가이드 - 실습 코드 1: RAG 시스템 구현 (LangChain)

- Tutorial ID: `expand-rag-fundamentals`
- Tutorial: RAG: 검색증강생성 완전 가이드
- Section ID: `expand-rag-fundamentals-code-1`
- Section: 실습 코드 1: RAG 시스템 구현 (LangChain)

> 이 노트북은 원본 실습 코드를 **처음 RAG를 배우는 사람** 기준으로 다시 작성한 상세 설명판입니다.
> 코드 한 줄 한 줄이 왜 필요한지, 실행하면 실제로 어떤 값이 나오는지를 직접 실행한 결과와 함께 보여줍니다.

## 이 노트북에서 배우는 것

이 노트북을 끝까지 따라가면 아래 질문에 스스로 답할 수 있게 됩니다.

1. RAG(검색증강생성)가 **왜** 필요한가?
2. 문서를 왜 잘게 잘라야 하는가? (청킹)
3. "임베딩"이라는 것이 실제로 무엇을 하는가?
4. 벡터스토어는 왜 필요하고, 검색기(Retriever)는 무슨 일을 하는가?
5. 검색된 문서와 질문을 LLM에 어떻게 넘겨서 답을 받는가?
6. 위 과정을 **LangChain**으로 어떻게 구현하는가? (2026년 현재 버전 기준)

### 사전 준비물
- Python 기본 문법 (변수, 함수, 클래스 정도만 알면 충분합니다)
- 별도의 딥러닝 지식은 필요 없습니다. 필요한 개념은 등장할 때마다 그 자리에서 설명합니다.
- 마지막 "실전 버전" 코드를 직접 실행해보려면 OpenAI API 키(유료)가 있으면 좋지만, 없어도 이 노트북의 나머지 부분은 전부 이해하고 실행할 수 있도록 구성했습니다.

### 목차
1. RAG란 무엇인가?
2. 실습 예제 소개 (사내 규정 Q&A 봇)
3. 지식 베이스 문서 준비
4. 1단계 - 문서 로딩
5. 2단계 - 청킹(Chunking)
6. 3단계 - 임베딩(Embedding)
7. 4단계 - 벡터스토어(Vector Store)
8. 5단계 - 검색기(Retriever)
9. 6단계 - 프롬프트와 답변 생성
10. 7단계 - 실전 버전으로 업그레이드
11. 정리, 자주 하는 실수, 다음 단계

## 이 노트북을 읽는 법

- 이 노트북은 "정답 코드를 한 번 실행하고 끝"이 아니라, **하나의 질문이 답이 되어 나오기까지 데이터가 어떤 모양으로 변해가는지**를 한 단계씩 눈으로 확인하기 위한 실습 노트입니다.
- 모든 코드 셀은 아래 원칙을 따릅니다.
  - 새로운 함수/클래스가 등장하기 전에 **바로 위 마크다운 셀에서 먼저 설명**합니다.
  - 인터넷 연결이나 API 키 없이 실행 가능한 코드는 **실제로 실행한 결과**를 그대로 출력에 남겨두었습니다. (여러분이 그대로 실행해도 같은 결과가 나옵니다)
  - OpenAI API 키나 대용량 모델 다운로드가 필요한 코드는 **"(참고, 미실행)"** 이라고 표시했습니다. 구조는 동일하되 실제 실행에는 키/인터넷이 필요합니다.
- 읽는 순서: ①먼저 왜 이 단계가 필요한지 읽기 → ②코드를 읽기 → ③출력 결과를 보며 "아, 이렇게 바뀌는구나"를 확인 → ④다음 단계로 이동.

## 1. RAG란 무엇인가?

### LLM만 사용했을 때의 한계

ChatGPT나 Claude 같은 LLM(거대 언어모델)은 아주 똑똑하지만, 아래와 같은 한계가 있습니다.

- **모르는 정보는 답할 수 없습니다.** 예를 들어 "우리 회사 재택근무 규정이 뭐야?"라는 질문에는 답할 수 없습니다. 그 회사의 내부 규정을 학습한 적이 없기 때문입니다.
- **최신 정보를 반영하지 못합니다.** 모델은 특정 시점까지의 데이터로 학습되었기 때문에, 그 이후에 생긴 정보는 알지 못합니다.
- **환각(Hallucination)이 발생할 수 있습니다.** 모르는 내용도 그럴듯하게 지어내서 답하는 경우가 있습니다.

### 오픈북 시험에 비유하기

RAG(Retrieval-Augmented Generation, 검색증강생성)는 이 문제를 **오픈북 시험**처럼 풀어냅니다.

- 클로즈북 시험(그냥 LLM에게 질문) = 학생이 머릿속에 외운 지식만으로 답하는 것
- 오픈북 시험(RAG) = 시험 중에 **참고 자료(교과서)를 펼쳐서 관련 페이지를 찾아본 뒤** 답을 쓰는 것

즉, RAG는 질문이 들어오면 (1) 관련 있는 문서를 먼저 찾아오고(Retrieval), (2) 그 문서를 참고해서 LLM이 답을 생성(Generation)하도록 만드는 구조입니다. 그래서 이름이 "검색(Retrieval) + 증강(Augmented) + 생성(Generation)"입니다.

### 전체 파이프라인 한눈에 보기

```
[사용자 질문]
     │
     ▼
① 질문을 벡터(숫자 배열)로 변환
     │
     ▼
② 미리 만들어둔 문서 벡터들 중 가장 비슷한 것을 검색 (벡터스토어)
     │
     ▼
③ 검색된 문서 + 원래 질문을 하나의 프롬프트로 결합
     │
     ▼
④ LLM이 그 프롬프트를 보고 답변 생성
     │
     ▼
[최종 답변] (+ 참고한 문서 출처)
```

이 노트북은 이 그림의 ①~④를 LangChain 코드로 하나씩 직접 만들어 봅니다. 그리고 그 전에, 문서들을 미리 벡터로 바꿔서 검색 가능한 상태로 만들어 두는 "준비 작업"(문서 로딩 → 청킹 → 임베딩 → 인덱싱)도 함께 다룹니다.

## 2. 이 노트북에서 만들 예제: 사내 규정 Q&A 봇

이해를 돕기 위해, 아주 구체적인 시나리오 하나를 끝까지 사용합니다.

> 가상의 회사 **"그린라이트 주식회사"**가 있습니다. 이 회사는 사내 규정(휴가, 재택근무, 경조사 지원 등)을 담은 문서를 가지고 있고, 직원들이 "재택근무는 얼마나 할 수 있나요?" 같은 질문을 하면 규정 문서를 찾아서 답해주는 챗봇을 만들려고 합니다.

이런 "사내 문서 기반 Q&A"는 실제로 RAG가 가장 많이 쓰이는 대표적인 사례입니다. 원본 노트북에서는 `knowledge_base.txt`라는, 실제로는 존재하지 않는 파일을 불러오는 코드만 있었습니다. 이 노트북에서는 그 파일을 **우리가 직접 채워서** 처음부터 끝까지 실행 가능하게 만듭니다.

## 실습 환경 준비하기

아래 라이브러리가 필요합니다. 터미널(또는 Colab 셀)에서 한 번만 설치하면 됩니다.

- `langchain-core`, `langchain-text-splitters` : LangChain의 핵심 부품, 텍스트 분할기
- `langchain-chroma`, `chromadb` : 벡터스토어(Chroma)
- `langchain-huggingface` : 무료 오픈소스 임베딩 모델을 쓰기 위한 패키지
- `langchain-openai` : OpenAI의 채팅 모델(GPT)을 쓰기 위한 패키지
- `scikit-learn` : 이 노트북의 "연습용" 임베딩(TF-IDF)을 만드는 데 사용

> 참고: 이 노트북을 만드는 시점(2026년) 기준으로, 예전에 많이 쓰이던 `langchain-community` 패키지는 유지보수가 종료(sunset)된다고 공식 발표되었습니다. 지금 당장 못 쓰게 되는 것은 아니지만, `TextLoader`처럼 아주 단순한 기능은 이 노트북처럼 **직접 코드 몇 줄로 대체**하는 방법도 함께 알아두면 좋습니다.

In [ ]:
# 이 노트북을 실행하기 전에 아래 라이브러리를 설치하세요.
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
!pip install langchain-core langchain-text-splitters langchain-chroma chromadb
!pip install langchain-huggingface langchain-openai scikit-learn

## 3. 지식 베이스 문서 준비하기

RAG를 실습하려면 "검색 대상이 되는 문서"가 있어야 합니다. 실제 서비스라면 사내 위키, 매뉴얼 PDF, 고객 상담 기록 등이 여기에 해당하겠지만, 이 노트북에서는 우리가 직접 짧은 사내 규정 텍스트를 만들어서 `knowledge_base.txt` 파일로 저장하겠습니다.

아래 텍스트에는 일부러 서로 다른 주제(휴가, 재택근무, 경조사, 교육비, 동호회)를 다섯 문단으로 나누어 담았습니다. 뒤에서 "청킹"을 배울 때 이 문단 구분이 어떻게 활용되는지 확인할 것입니다.

In [1]:
knowledge_base_text = """그린라이트 주식회사 사내 규정 안내

1. 근무 시간 및 휴가 규정
그린라이트 주식회사의 표준 근무 시간은 오전 9시부터 오후 6시까지이며, 점심 시간은 1시간입니다. 직원은 입사 첫 해에 연차 유급휴가 15일을 부여받으며, 근속 연수가 2년 늘어날 때마다 1일씩 추가되어 최대 25일까지 늘어납니다. 연차는 반차(0.5일) 단위로도 사용할 수 있으며, 사용하지 않은 연차는 다음 해로 최대 5일까지 이월할 수 있습니다. 연차 사용을 원할 경우 최소 사용 1일 전까지 팀장에게 사내 시스템을 통해 신청해야 합니다.

2. 재택근무 정책
그린라이트 주식회사는 주 2회까지 재택근무를 허용합니다. 재택근무를 하려는 직원은 전날 오후 6시까지 팀장의 승인을 받아야 하며, 재택근무 중에도 근무 시간 동안 사내 메신저로 연락이 가능해야 합니다. 신입사원은 입사 후 3개월의 수습 기간에는 재택근무를 사용할 수 없으며, 수습 기간이 끝난 이후부터 신청이 가능합니다. 재택근무 시에도 정기 회의에는 화상으로 참석해야 합니다.

3. 경조사 지원 규정
직원 본인이 결혼하는 경우 5일의 경조휴가와 축의금 100만원이 지급됩니다. 직원의 부모님 또는 배우자의 부모님이 돌아가신 경우 5일의 경조휴가와 조의금 50만원이 지급됩니다. 직원 본인의 자녀가 태어난 경우에는 배우자 출산휴가 10일이 별도로 부여됩니다. 경조사 지원을 받으려면 관련 증빙 서류를 인사팀에 제출해야 합니다.

4. 교육비 지원 제도
직무 관련 교육이나 자격증 취득을 위한 강의 수강료의 70퍼센트를 연간 최대 100만원까지 지원합니다. 교육비 지원을 받으려면 사전에 팀장과 인사팀의 승인을 받아야 하며, 교육이 끝난 뒤 수료증과 영수증을 제출해야 최종 지급됩니다. 어학 관련 교육은 연간 최대 50만원까지 별도로 지원됩니다.

5. 사내 동호회 지원
네 명 이상의 직원이 모여 동호회를 만들면 분기마다 20만원의 활동비를 지원받을 수 있습니다. 지원을 받으려면 동호회 명단과 활동 계획서를 총무팀에 제출해야 하며, 반기마다 활동 보고서를 제출해야 지원이 계속됩니다. 현재 사내에는 등산 동호회, 보드게임 동호회, 러닝 동호회가 운영되고 있습니다.
"""

with open("knowledge_base.txt", "w", encoding="utf-8") as f:
    f.write(knowledge_base_text)

print(f"knowledge_base.txt 파일이 생성되었습니다. (총 {len(knowledge_base_text)}자)")
print("--- 내용 미리보기 ---")
print(knowledge_base_text[:120] + "...")

knowledge_base.txt 파일이 생성되었습니다. (총 1066자)
--- 내용 미리보기 ---
그린라이트 주식회사 사내 규정 안내

1. 근무 시간 및 휴가 규정
그린라이트 주식회사의 표준 근무 시간은 오전 9시부터 오후 6시까지이며, 점심 시간은 1시간입니다. 직원은 입사 첫 해에 연차 유급휴가 15일을 부...


## 4. 1단계 - 문서 로딩 (Document Loading)

"로딩(loading)"이란 디스크에 있는 파일(txt, pdf, csv 등)을 읽어서, LangChain이 다룰 수 있는 공통 형태인 **`Document` 객체**로 바꾸는 작업입니다.

`Document` 객체는 아주 단순합니다. 아래 두 가지만 가지고 있습니다.

- `page_content` : 실제 텍스트 내용 (문자열)
- `metadata` : 이 문서에 대한 부가 정보 (예: 어떤 파일에서 왔는지, 몇 페이지인지 등을 담는 딕셔너리)

원본 노트북에서는 `langchain.document_loaders.TextLoader`라는 헬퍼 클래스를 사용했습니다. 이 클래스가 내부적으로 하는 일은 사실 "파일을 열어서 읽고, `Document`로 감싸는 것" 뿐입니다. 그래서 여기서는 그 내부 동작을 그대로 직접 작성해서, `Document`가 정확히 무엇인지 눈으로 확인해보겠습니다. (파일 종류가 pdf, csv, 웹페이지 등으로 다양해지면 그때는 전용 로더를 쓰는 것이 훨씬 편리합니다. 지금은 원리를 보기 위한 가장 단순한 경우입니다.)

In [2]:
from langchain_core.documents import Document

# 1. 파일을 그냥 파이썬으로 읽습니다.
with open("knowledge_base.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 2. LangChain이 이해할 수 있는 Document 객체로 감쌉니다.
#    metadata에는 "이 내용이 어디서 왔는지"를 적어두면, 나중에 답변의 출처를 표시할 때 유용합니다.
docs = [Document(page_content=raw_text, metadata={"source": "knowledge_base.txt"})]

print(f"불러온 문서 개수: {len(docs)}")
print(f"문서 객체 타입: {type(docs[0]).__name__}")
print(f"전체 글자 수: {len(docs[0].page_content)}자")
print(f"메타데이터: {docs[0].metadata}")
print(f"\n앞부분 미리보기:\n{docs[0].page_content[:80]}...")

불러온 문서 개수: 1
문서 객체 타입: Document
전체 글자 수: 1066자
메타데이터: {'source': 'knowledge_base.txt'}

앞부분 미리보기:
그린라이트 주식회사 사내 규정 안내

1. 근무 시간 및 휴가 규정
그린라이트 주식회사의 표준 근무 시간은 오전 9시부터 오후 6시까지이며, 점...


> **참고 (TextLoader를 쓰고 싶다면)**: 다른 강의나 문서에서는 여전히 아래처럼 `TextLoader`를 쓰는 코드를 많이 보게 될 것입니다. 결과는 위 코드와 동일합니다.
> ```python
> from langchain_community.document_loaders import TextLoader
> loader = TextLoader("knowledge_base.txt", encoding="utf-8")
> docs = loader.load()
> ```
> 다만 `langchain_community` 패키지는 2026년에 유지보수 종료가 공식 발표되었기 때문에, 새 프로젝트에서는 위처럼 직접 `Document`를 만들거나, 각 서비스 전용 패키지(예: PDF는 `pypdf`, 웹페이지는 별도 로더 패키지)를 쓰는 방향을 권장합니다.

## 5. 2단계 - 청킹(Chunking)

### 왜 문서를 통째로 쓰지 않고 잘게 자르나요?

두 가지 이유가 있습니다.

1. **LLM에는 한 번에 넣을 수 있는 텍스트 길이(컨텍스트 윈도우) 제한이 있습니다.** 문서 전체가 수백 페이지라면 통째로 넣을 수 없습니다.
2. **검색 정밀도가 떨어집니다.** 질문 하나에 관련된 내용은 보통 문서의 아주 일부분입니다. 문서 전체를 하나의 덩어리로 취급하면, "재택근무" 질문에도 휴가·경조사·교육비 내용이 전부 섞여서 딸려 나오게 됩니다.

그래서 문서를 적당한 크기의 "청크(chunk, 조각)"로 미리 잘라두고, 질문과 가장 관련 있는 청크만 찾아서 쓰는 것이 RAG의 기본 전략입니다.

### chunk_size와 chunk_overlap

- `chunk_size` : 청크 하나의 최대 글자 수
- `chunk_overlap` : 바로 앞 청크와 겹치게 남겨두는 글자 수

왜 겹치는 부분(overlap)을 두나요? 문장이 청크 경계에서 뚝 끊기면, 그 경계에 걸쳐 있는 내용이 앞뒤 어느 청크에서도 온전히 검색되지 않을 수 있습니다. 조금씩 겹치게 잘라두면 이런 손실을 줄일 수 있습니다.

말로만 들으면 헷갈리니, 아주 짧은 문장으로 직접 확인해보겠습니다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_sentence = "인공지능은 컴퓨터가 인간처럼 학습하고 판단하도록 만드는 기술입니다. 최근에는 대규모 언어모델이 다양한 분야에서 활용되고 있습니다."
print(f"원본 문장: {sample_sentence}")
print(f"원본 길이: {len(sample_sentence)}자\n")

# 이해를 돕기 위해 아주 작은 값으로 설정합니다. (실제 서비스에서는 훨씬 큰 값을 씁니다)
tiny_splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=10)
tiny_chunks = tiny_splitter.split_text(sample_sentence)

print(f"만들어진 청크 개수: {len(tiny_chunks)}\n")
for i, chunk in enumerate(tiny_chunks):
    print(f"[청크 {i+1}] (길이 {len(chunk)}자) {chunk!r}")

원본 문장: 인공지능은 컴퓨터가 인간처럼 학습하고 판단하도록 만드는 기술입니다. 최근에는 대규모 언어모델이 다양한 분야에서 활용되고 있습니다.
원본 길이: 72자

만들어진 청크 개수: 3

[청크 1] (길이 37자) '인공지능은 컴퓨터가 인간처럼 학습하고 판단하도록 만드는 기술입니다.'
[청크 2] (길이 35자) '기술입니다. 최근에는 대규모 언어모델이 다양한 분야에서 활용되고'
[청크 3] (길이 15자) '분야에서 활용되고 있습니다.'


출력을 잘 보면, **청크 1의 끝부분("기술입니다.")이 청크 2의 시작 부분에 다시 등장**하는 것을 볼 수 있습니다. 이게 바로 `chunk_overlap=10`이 하는 일입니다. 또한 `RecursiveCharacterTextSplitter`라는 이름처럼, 이 분할기는 문단(`\n\n`) → 줄바꿈(`\n`) → 띄어쓰기(` `) → 글자 순서로, 가능하면 문장이나 단어를 중간에서 끊지 않으려고 노력합니다. (그래서 "이름"이 "Recursive"입니다: 큰 단위부터 순서대로 시도합니다.)

이제 실제 지식 베이스 문서에 적용해보겠습니다. 이번에는 실습용 문서 크기에 맞춰 `chunk_size=300`, `chunk_overlap=50`을 사용합니다. (실제 서비스에서는 문서 성격에 따라 보통 500~1500자 정도를 많이 사용합니다. 문서가 짧을수록, 그리고 질문이 짧고 명확할수록 작은 chunk_size가 유리한 경우가 많습니다.)

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,     # 실습 문서 크기에 맞춘 값 (실제 서비스에서는 500~1500자를 많이 사용)
    chunk_overlap=50,
)
chunks = splitter.split_documents(docs)

print(f"총 청크 개수: {len(chunks)}\n")
for i, c in enumerate(chunks):
    preview = c.page_content.replace("\n", " ").strip()
    print(f"--- 청크 {i+1} (길이 {len(c.page_content)}자) ---")
    print(preview)
    print()

총 청크 개수: 5

--- 청크 1 (길이 287자) ---
그린라이트 주식회사 사내 규정 안내  1. 근무 시간 및 휴가 규정 그린라이트 주식회사의 표준 근무 시간은 오전 9시부터 오후 6시까지이며, 점심 시간은 1시간입니다. 직원은 입사 첫 해에 연차 유급휴가 15일을 부여받으며, 근속 연수가 2년 늘어날 때마다 1일씩 추가되어 최대 25일까지 늘어납니다. 연차는 반차(0.5일) 단위로도 사용할 수 있으며, 사용하지 않은 연차는 다음 해로 최대 5일까지 이월할 수 있습니다. 연차 사용을 원할 경우 최소 사용 1일 전까지 팀장에게 사내 시스템을 통해 신청해야 합니다.

--- 청크 2 (길이 222자) ---
2. 재택근무 정책 그린라이트 주식회사는 주 2회까지 재택근무를 허용합니다. 재택근무를 하려는 직원은 전날 오후 6시까지 팀장의 승인을 받아야 하며, 재택근무 중에도 근무 시간 동안 사내 메신저로 연락이 가능해야 합니다. 신입사원은 입사 후 3개월의 수습 기간에는 재택근무를 사용할 수 없으며, 수습 기간이 끝난 이후부터 신청이 가능합니다. 재택근무 시에도 정기 회의에는 화상으로 참석해야 합니다.

--- 청크 3 (길이 194자) ---
3. 경조사 지원 규정 직원 본인이 결혼하는 경우 5일의 경조휴가와 축의금 100만원이 지급됩니다. 직원의 부모님 또는 배우자의 부모님이 돌아가신 경우 5일의 경조휴가와 조의금 50만원이 지급됩니다. 직원 본인의 자녀가 태어난 경우에는 배우자 출산휴가 10일이 별도로 부여됩니다. 경조사 지원을 받으려면 관련 증빙 서류를 인사팀에 제출해야 합니다.

--- 청크 4 (길이 176자) ---
4. 교육비 지원 제도 직무 관련 교육이나 자격증 취득을 위한 강의 수강료의 70퍼센트를 연간 최대 100만원까지 지원합니다. 교육비 지원을 받으려면 사전에 팀장과 인사팀의 승인을 받아야 하며, 교육이 끝난 뒤 수료증과 영수증을 제출해야 최종 지급됩니다. 어학 관련 교육은 연간 최대 50만원까지 별도로 지원됩니다.

--- 청크 5 

지식 베이스에 있던 다섯 개의 문단이 정확히 다섯 개의 청크로 나뉜 것을 볼 수 있습니다. 각 문단이 `chunk_size=300`보다 짧기 때문에, 분할기가 문단 사이의 빈 줄(`\n\n`)에서 깔끔하게 잘라준 것입니다. 만약 문단 하나가 300자보다 길었다면, 그 문단 안에서도 다시 문장이나 띄어쓰기 단위로 나뉘었을 것입니다.

## 6. 3단계 - 임베딩(Embedding)

### 임베딩이 하는 일

컴퓨터는 "재택근무"와 "원격근무"가 비슷한 말이라는 것을 문자 그대로는 알지 못합니다. 그래서 텍스트를 **의미가 비슷하면 서로 가까운 위치에 놓이는 숫자 배열(벡터)** 로 바꾸는 작업이 필요합니다. 이 변환을 **임베딩(embedding)** 이라고 부릅니다.

비유하자면, 임베딩은 모든 문장에 **지도 위의 좌표**를 하나씩 찍어주는 것과 같습니다. 의미가 비슷한 문장은 지도에서 가까운 위치에 찍히고, 관련 없는 문장은 멀리 떨어진 위치에 찍힙니다. 그러면 "질문과 가장 비슷한 문서 찾기"는 "지도에서 질문의 좌표와 가장 가까운 문서 좌표 찾기"라는 계산 문제로 바뀝니다.

실제 서비스에서는 이 좌표를 딥러닝 모델(예: 문장을 이해하도록 학습된 신경망)이 만들어줍니다. 하지만 원리를 먼저 이해하기 위해, 훨씬 단순한 통계 기법인 **TF-IDF**로 "연습용" 임베딩을 직접 만들어보겠습니다.

### TF-IDF란?

TF-IDF는 "이 단어(조각)가 이 문서에는 자주 나오지만, 전체 문서 집합에서는 드물게 나올수록 그 단어에 높은 점수를 준다"는 아주 직관적인 통계 방식입니다. 예를 들어 "재택근무"라는 단어가 청크 2에서만 반복해서 등장한다면, TF-IDF는 그 단어에 높은 가중치를 주어서 청크 2를 대표하는 특징으로 삼습니다.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_texts = [c.page_content for c in chunks]

# 가장 기본적인 방식: 단어(공백) 단위로 쪼개서 벡터를 만듭니다.
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunk_texts)

query = "재택근무는 얼마나 할 수 있나요?"
query_vector = vectorizer.transform([query])

similarities = cosine_similarity(query_vector, chunk_vectors)[0]

print(f"질문: {query}\n")
for i, score in enumerate(similarities):
    print(f"청크 {i+1} 유사도: {score:.4f}")

print(f"\n-> 전부 0에 가깝습니다. 왜 그럴까요?")

질문: 재택근무는 얼마나 할 수 있나요?

청크 1 유사도: 0.0000
청크 2 유사도: 0.0000
청크 3 유사도: 0.0000
청크 4 유사도: 0.0000
청크 5 유사도: 0.0000

-> 전부 0에 가깝습니다. 왜 그럴까요?


결과가 이상합니다. 분명 청크 2(재택근무 정책)가 질문과 관련이 큰데, 유사도가 전부 0에 가깝게 나왔습니다.

**원인은 한국어의 조사입니다.** 질문에는 "재택근무**는**"이라는 형태로 들어있고, 문서에는 "재택근무**를**", "재택근무" 등 다른 조사가 붙은 형태로 등장합니다. 영어라면 "remote work"가 어디에 있든 똑같은 단어이지만, 한국어는 조사가 단어 바로 뒤에 붙어버리기 때문에 **띄어쓰기 기준으로 자르면 "재택근무는"과 "재택근무를"이 완전히 다른 단어로 취급**되어 버립니다.

이런 경우 흔히 쓰는 간단한 해결책은, 단어 단위가 아니라 **글자 2~4개씩 겹쳐서 잘라낸 조각(character n-gram)** 단위로 비교하는 것입니다. 이렇게 하면 "재택근무는"과 "재택근무를"도 "재택근무"라는 공통 글자 조각을 상당수 공유하게 되어, 조사가 달라도 비슷한 벡터가 만들어집니다.

In [6]:
# analyzer="char_wb" : 단어가 아니라 '글자 조각' 단위로 비교
# ngram_range=(2, 4) : 2~4글자씩 겹쳐서 잘라냄 (예: "재택근무" -> "재택", "택근", "근무", "재택근", ...)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
chunk_vectors = vectorizer.fit_transform(chunk_texts)
print(f"청크 벡터 행렬의 shape: {chunk_vectors.shape}  (청크 개수, 전체 글자조각 종류 수)")

query = "재택근무는 얼마나 할 수 있나요?"
query_vector = vectorizer.transform([query])
similarities = cosine_similarity(query_vector, chunk_vectors)[0]

print(f"\n질문: {query}\n")
for i, score in enumerate(similarities):
    print(f"청크 {i+1} 유사도: {score:.4f}")

best_idx = similarities.argmax()
print(f"\n가장 유사도가 높은 청크 -> 청크 {best_idx+1}")
print(chunk_texts[best_idx][:120].replace(chr(10), " "))

# 다른 질문으로도 확인해봅니다.
query2 = "결혼하면 며칠 쉴 수 있나요?"
query_vector2 = vectorizer.transform([query2])
similarities2 = cosine_similarity(query_vector2, chunk_vectors)[0]
best_idx2 = similarities2.argmax()
print(f"\n질문2: {query2}")
for i, score in enumerate(similarities2):
    print(f"청크 {i+1} 유사도: {score:.4f}")
print(f"가장 유사도가 높은 청크 -> 청크 {best_idx2+1}")

청크 벡터 행렬의 shape: (5, 1352)  (청크 개수, 전체 글자조각 종류 수)

질문: 재택근무는 얼마나 할 수 있나요?

청크 1 유사도: 0.0772
청크 2 유사도: 0.5629
청크 3 유사도: 0.0112
청크 4 유사도: 0.0215
청크 5 유사도: 0.0325

가장 유사도가 높은 청크 -> 청크 2
2. 재택근무 정책 그린라이트 주식회사는 주 2회까지 재택근무를 허용합니다. 재택근무를 하려는 직원은 전날 오후 6시까지 팀장의 승인을 받아야 하며, 재택근무 중에도 근무 시간 동안 사내 메신저로 연락이 가능해야 합

질문2: 결혼하면 며칠 쉴 수 있나요?
청크 1 유사도: 0.0463
청크 2 유사도: 0.0226
청크 3 유사도: 0.0934
청크 4 유사도: 0.0175
청크 5 유사도: 0.0480
가장 유사도가 높은 청크 -> 청크 3


이번에는 두 질문 모두 정확한 청크(재택근무 → 청크 2, 결혼 → 청크 3)를 잘 찾아냈습니다. 실제 딥러닝 기반 임베딩(HuggingFace, OpenAI 등)은 이보다 훨씬 정교한 방식으로 "의미"를 이해하지만, **"질문과 문서를 같은 방식으로 벡터화해서, 벡터 사이의 거리(유사도)로 관련성을 판단한다"** 는 핵심 아이디어는 동일합니다.

### LangChain과 연결하기 위한 Embeddings 클래스 만들기

LangChain의 벡터스토어(다음 단계에서 배울 Chroma 등)는 아무 임베딩이나 바로 쓸 수 없고, 정해진 규칙(인터페이스)을 따르는 객체를 요구합니다. 그 규칙은 `langchain_core.embeddings.Embeddings`를 상속받아 아래 두 메서드를 구현하는 것입니다.

- `embed_documents(texts)` : 여러 문서를 한꺼번에 벡터 리스트로 변환
- `embed_query(text)` : 질문 하나를 벡터로 변환

방금 만든 TF-IDF 방식을 이 규칙에 맞게 클래스로 감싸보겠습니다.

In [7]:
from langchain_core.embeddings import Embeddings

class PracticeEmbeddings(Embeddings):
    """
    문자 조각(char n-gram) 기반 TF-IDF로 벡터를 만드는 '연습용' 임베딩입니다.
    실제 서비스에서는 이 자리에 HuggingFaceEmbeddings, OpenAIEmbeddings 같은
    딥러닝 기반 임베딩을 사용합니다. (7단계에서 그대로 바꿔볼 것입니다)
    """
    def __init__(self, corpus):
        # 전체 문서(코퍼스)를 기준으로 어떤 글자 조각이 있는지 미리 학습해둡니다.
        self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
        self.vectorizer.fit(corpus)

    def embed_documents(self, texts):
        return self.vectorizer.transform(texts).toarray().tolist()

    def embed_query(self, text):
        # 질문도 문서와 '같은' vectorizer로 변환해야 같은 좌표계에서 비교할 수 있습니다.
        return self.vectorizer.transform([text]).toarray()[0].tolist()

practice_embeddings = PracticeEmbeddings(corpus=chunk_texts)

sample_vector = practice_embeddings.embed_query("연차는 며칠 주어지나요?")
print(f"벡터의 전체 차원 수: {len(sample_vector)}")

import numpy as np
arr = np.array(sample_vector)
nz = np.nonzero(arr)[0]
print(f"0이 아닌 값의 개수: {len(nz)}개  (나머지는 전부 0인 희소(sparse) 벡터입니다)")
print("0이 아닌 값 미리보기 (인덱스: 값):")
for i in nz[:6]:
    print(f"  {i}: {arr[i]:.4f}")

벡터의 전체 차원 수: 1352
0이 아닌 값의 개수: 10개  (나머지는 전부 0인 희소(sparse) 벡터입니다)
0이 아닌 값 미리보기 (인덱스: 값):
  231: 0.2308
  238: 0.3447
  240: 0.3447
  308: 0.2781
  614: 0.1942
  958: 0.3447


1352차원 벡터인데 실제 값이 들어있는 곳은 단 10곳뿐입니다. 이런 벡터를 **희소 벡터(sparse vector)**라고 합니다. 나중에 살펴볼 HuggingFace/OpenAI 같은 딥러닝 임베딩은 보통 384~1536차원 정도의 **훨씬 작지만 거의 모든 값이 채워진(dense) 벡터**를 만듭니다. 차원 수는 훨씬 적어도, 각 차원이 "의미"의 어떤 측면을 더 응축해서 표현하기 때문에 훨씬 똑똑하게 비교할 수 있습니다.

## 7. 4단계 - 벡터스토어(Vector Store)에 저장하기

문서가 5개뿐이라면 질문 벡터와 5번 비교해서 가장 가까운 것을 고르면 됩니다. 하지만 문서가 수만~수백만 개라면 매번 전체를 하나씩 비교하는 것은 너무 느립니다.

**벡터스토어(vector store)** 는 이런 대량의 벡터를 미리 저장해두고, 질문이 들어오면 빠르게 "가장 가까운 벡터 top-k개"를 찾아주는 전용 데이터베이스입니다. 도서관에 비유하면, 책을 서가에 아무렇게나 꽂아두지 않고 **분류 체계(색인)** 에 따라 정리해두어서, 사서가 원하는 책을 빠르게 찾아줄 수 있게 하는 것과 같은 역할입니다.

이 노트북에서는 로컬 환경에서 간단히 쓸 수 있는 오픈소스 벡터스토어인 **Chroma**를 사용합니다. 앞에서 만든 `chunks`(문서 조각들)와 `practice_embeddings`(연습용 임베딩)를 넘겨주기만 하면, Chroma가 내부적으로 모든 청크를 벡터로 변환해서 저장해줍니다.

In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=practice_embeddings,
    collection_name="greenlight_hr_practice",
)

print("벡터스토어 생성 완료")
stored = vectorstore.get()
print(f"저장된 청크 수: {len(stored['ids'])}")

벡터스토어 생성 완료
저장된 청크 수: 5


`Chroma.from_documents`를 호출하는 순간, 내부적으로는 이런 일이 일어납니다.

1. 각 청크의 `page_content`를 `practice_embeddings.embed_documents(...)`에 넣어서 벡터로 변환
2. 그 벡터들을 청크 원문·메타데이터와 함께 데이터베이스에 저장

즉, 지금까지 우리가 배운 "문서 → 벡터" 변환 과정을, Chroma가 내부에서 자동으로 호출해준 것뿐입니다.

## 8. 5단계 - 검색기(Retriever) 만들기

벡터스토어를 "질문 하나를 넣으면 관련 문서를 돌려주는 함수"처럼 쓸 수 있게 감싼 것이 **검색기(Retriever)**입니다. `as_retriever()`를 호출할 때 `k` 값으로 "몇 개까지 가져올지"를 정합니다.

- `k`가 너무 작으면: 정답에 필요한 청크가 빠질 수 있습니다.
- `k`가 너무 크면: 관련 없는 내용까지 LLM에게 전달되어, 답변 품질이 떨어지거나 비용이 늘어날 수 있습니다.

보통 3~5 사이 값에서 시작해 실험을 통해 조정합니다. 여기서는 예제 문서가 작으므로 `k=2`로 해보겠습니다.

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

test_queries = ["재택근무는 얼마나 할 수 있나요?", "결혼하면 며칠 쉴 수 있나요?"]
for q in test_queries:
    print(f"질문: {q}")
    found = retriever.invoke(q)
    for i, d in enumerate(found):
        preview = d.page_content[:50].replace(chr(10), " ")
        print(f"  검색된 청크 {i+1}: {preview}...")
    print()

질문: 재택근무는 얼마나 할 수 있나요?
  검색된 청크 1: 2. 재택근무 정책 그린라이트 주식회사는 주 2회까지 재택근무를 허용합니다. 재택근무를 하...
  검색된 청크 2: 그린라이트 주식회사 사내 규정 안내  1. 근무 시간 및 휴가 규정 그린라이트 주식회사의 ...

질문: 결혼하면 며칠 쉴 수 있나요?
  검색된 청크 1: 3. 경조사 지원 규정 직원 본인이 결혼하는 경우 5일의 경조휴가와 축의금 100만원이 지...
  검색된 청크 2: 5. 사내 동호회 지원 네 명 이상의 직원이 모여 동호회를 만들면 분기마다 20만원의 활동...


두 질문 모두 관련 있는 청크가 1순위로 검색되는 것을 확인할 수 있습니다. 이제 "질문 → 관련 문서 찾기"까지는 완성되었습니다. 남은 것은 이 문서를 LLM에게 넘겨서 실제 답변 문장을 만들어내는 부분입니다.

## 9. 6단계 - 프롬프트 구성과 답변 생성

이제 마지막 조각입니다. 검색된 청크들을 질문과 함께 하나의 프롬프트(LLM에게 보낼 지시문)로 묶고, 그 프롬프트를 LLM에게 전달해서 답을 받아야 합니다.

이 과정을 **"stuff(채워 넣기)" 방식**이라고 부릅니다. 검색된 문서들을 그대로 프롬프트의 빈칸에 채워 넣는다는 뜻입니다. (문서가 아주 많을 때는 요약 후 채우기 등 다른 방식도 있지만, 가장 기본적이고 널리 쓰이는 방식이 바로 이 stuff 방식입니다.)

먼저 프롬프트 템플릿을 만들어서, `{context}`(검색된 문서)와 `{input}`(질문) 자리에 실제 값이 어떻게 채워지는지 확인해보겠습니다.

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "다음은 회사 규정에서 찾은 참고 자료입니다. 참고 자료에 있는 내용만 바탕으로 질문에 답하세요.\n"
    "참고 자료에 없는 내용이면 \"규정에서 찾을 수 없습니다\"라고 답하세요.\n\n"
    "[참고 자료]\n{context}\n\n[질문]\n{input}\n\n[답변]"
)

# 프롬프트가 실제로 어떻게 채워지는지 미리 값을 넣어서 확인해봅니다.
example_prompt = prompt.invoke({
    "context": "재택근무는 주 2회까지 가능합니다.",
    "input": "재택근무는 얼마나 할 수 있나요?",
})
print(example_prompt.to_string())

Human: 다음은 회사 규정에서 찾은 참고 자료입니다. 참고 자료에 있는 내용만 바탕으로 질문에 답하세요.
참고 자료에 없는 내용이면 "규정에서 찾을 수 없습니다"라고 답하세요.

[참고 자료]
재택근무는 주 2회까지 가능합니다.

[질문]
재택근무는 얼마나 할 수 있나요?

[답변]


`{context}`와 `{input}` 자리에 실제 텍스트가 채워진 것을 볼 수 있습니다. 앞에 "Human:"이 자동으로 붙는 이유는, 채팅 모델(ChatOpenAI 등)은 "누가 말한 내용인지" 역할이 표시된 메시지 형태를 기대하기 때문입니다. `ChatPromptTemplate.from_template(...)`은 기본적으로 이 전체를 "사람이 말한 메시지" 하나로 만들어줍니다.

### 왜 진짜 LLM 대신 "가짜 모델"로 먼저 확인하나요?

실제 LLM(OpenAI GPT 등)을 호출하려면 API 키가 있어야 하고, 호출할 때마다 비용이 발생합니다. 그런데 지금 우리가 확인하고 싶은 것은 "답변의 품질"이 아니라 **"검색 → 프롬프트 구성 → 모델 호출 → 결과 파싱"이라는 배관(파이프라인)이 코드 오류 없이 잘 연결되는지**입니다.

이럴 때는 LangChain이 테스트용으로 제공하는 `FakeListChatModel`을 사용하면 좋습니다. 이 모델은 실제로 아무 것도 "생각"하지 않고, 미리 정해둔 답변을 그대로 돌려줍니다. API 키도, 인터넷 연결도, 비용도 필요 없습니다. 파이프라인 구조를 다 확인한 뒤에 진짜 모델로 바꾸면, 훨씬 안전하고 저렴하게 개발할 수 있습니다.

> 참고: 예전 LangChain(0.x 버전)에서는 `create_stuff_documents_chain` / `create_retrieval_chain` 같은 헬퍼 함수로 이 과정을 자동화했습니다. 이 노트북을 쓰는 시점(2026년) 기준으로 LangChain이 1.0으로 올라가면서 이런 체인 헬퍼들은 `langchain-classic`이라는 별도 패키지로 옮겨졌습니다. 그래서 여기서는 별도 패키지 없이도 동작하고, 무슨 일이 일어나는지 눈으로 볼 수 있도록 **직접 단계별로 연결**하는 방법을 보여드립니다.

In [11]:
from langchain_core.language_models import FakeListChatModel

# 실제 LLM처럼 동작하지만, 항상 미리 정해둔 답을 돌려주는 '가짜' 모델입니다.
fake_llm = FakeListChatModel(responses=[
    "회사 규정에 따르면 재택근무는 주 2회까지 가능하며, 전날 오후 6시까지 팀장의 승인을 받아야 합니다."
])

def format_docs(found_docs):
    # 검색된 여러 Document를 하나의 문자열로 이어붙입니다.
    return "\n\n".join(d.page_content for d in found_docs)

query = "재택근무는 얼마나 할 수 있나요?"

# ① 질문으로 관련 문서 검색
found_docs = retriever.invoke(query)

# ② 검색된 문서들을 하나의 문자열(context)로 합치기
context_text = format_docs(found_docs)

# ③ 프롬프트 템플릿에 context와 질문을 채워 넣기
filled_prompt = prompt.invoke({"context": context_text, "input": query})

# ④ (가짜) LLM에게 프롬프트를 전달해서 답변 받기
ai_message = fake_llm.invoke(filled_prompt)

print("검색된 청크 수:", len(found_docs))
print("답변:", ai_message.content)

검색된 청크 수: 2
답변: 회사 규정에 따르면 재택근무는 주 2회까지 가능하며, 전날 오후 6시까지 팀장의 승인을 받아야 합니다.


API 키 없이도 검색부터 답변 생성까지 전체 파이프라인이 오류 없이 연결된 것을 확인했습니다. 이 4단계(①~④)를 매번 이렇게 풀어서 쓰는 대신, LangChain의 파이프 연산자 `|`를 사용하면 한 번에 이어붙일 수도 있습니다. 아래는 완전히 동일한 동작을 하는 코드입니다.

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | fake_llm
    | StrOutputParser()
)

answer = rag_chain.invoke(query)
print("답변:", answer)

답변: 회사 규정에 따르면 재택근무는 주 2회까지 가능하며, 전날 오후 6시까지 팀장의 승인을 받아야 합니다.


`|`는 "왼쪽의 출력을 오른쪽의 입력으로 그대로 넘겨준다"는 뜻입니다. 위 코드는 다음을 의미합니다.

1. `retriever`로 검색한 결과를 `format_docs`에 넘겨 문자열로 합치고(→ `context`), 질문은 그대로 통과시켜서(→ `input`) 딕셔너리를 만든다
2. 그 딕셔너리를 `prompt`에 넣어 프롬프트를 완성한다
3. 완성된 프롬프트를 `fake_llm`에 넣어 답변을 받는다
4. `StrOutputParser`로 메시지 객체에서 순수 텍스트만 뽑아낸다

방금 직접 풀어서 쓴 ①~④ 코드와 정확히 같은 일을 하는 코드입니다. 처음에는 단계별 코드(방법 A)로 각 단계가 무엇을 하는지 이해하고, 익숙해지면 `|` 스타일(방법 B)로 간결하게 작성하는 것을 추천합니다.

## 10. 7단계 - 실전 서비스로 업그레이드하기

지금까지 만든 구조(문서 로딩 → 청킹 → 임베딩 → 벡터스토어 → 검색기 → 프롬프트 → 생성)는 실제 서비스와 **완전히 동일한 뼈대**입니다. 지금부터 할 일은 이 뼈대는 그대로 둔 채, 성능이 낮은 "연습용 부품" 두 가지만 성능 좋은 "실전용 부품"으로 갈아 끼우는 것입니다.

| 자리 | 연습용 (지금까지 사용) | 실전용 (지금부터 교체) |
|---|---|---|
| 임베딩 | `PracticeEmbeddings` (TF-IDF) | `HuggingFaceEmbeddings` 또는 `OpenAIEmbeddings` (딥러닝) |
| 생성 모델 | `FakeListChatModel` (고정 답변) | `ChatOpenAI` 등 실제 LLM |

아래 코드들은 **인터넷 연결(모델 다운로드)과 API 키가 필요하므로 이 노트북에서는 직접 실행하지 않았습니다.** 하지만 구조는 위에서 이미 검증한 것과 100% 동일하므로, 여러분의 환경에서 API 키만 준비되면 바로 실행됩니다.

### (참고, 미실행) 실전 임베딩 모델 사용하기

`sentence-transformers/all-MiniLM-L6-v2`는 문장을 384차원의 벡터로 바꿔주는, 널리 쓰이는 무료 오픈소스 임베딩 모델입니다. 처음 실행할 때 모델 파일(약 80MB)을 인터넷에서 자동으로 내려받습니다.

In [ ]:
# (참고, 미실행) 인터넷 연결이 필요하며 처음 실행 시 모델을 다운로드합니다.
from langchain_huggingface import HuggingFaceEmbeddings

real_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 사용법은 PracticeEmbeddings와 완전히 동일합니다 (같은 Embeddings 인터페이스를 따르기 때문입니다).
# vector = real_embeddings.embed_query("연차는 며칠 주어지나요?")
# print(len(vector))  # 384 가 출력됩니다.

### (참고, 미실행) API 키 설정하기

API 키는 **절대로 코드에 직접 적어서 저장/공유하지 마세요.** 코드가 유출되면 키도 함께 유출되어, 다른 사람이 여러분의 계정으로 요금을 발생시킬 수 있습니다. 아래처럼 실행할 때마다 안전하게 입력받거나, `.env` 파일 등 별도 설정 파일로 관리하는 것이 안전합니다.

In [ ]:
# (참고, 미실행)
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키를 입력하세요: ")

from langchain_openai import ChatOpenAI

real_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# temperature=0 : 같은 질문에는 최대한 같은(일관된) 답을 하도록, 무작위성을 최소화하는 설정입니다.

# 참고: OpenAI가 아니라 Anthropic의 Claude를 쓰고 싶다면 아래처럼 교체할 수 있습니다.
# from langchain_anthropic import ChatAnthropic
# real_llm = ChatAnthropic(model="claude-sonnet-5", temperature=0)

### (참고, 미실행) 전체 조합 - 최종 실전 파이프라인

아래는 지금까지 배운 내용을 모두 합친, 원본 노트북과 같은 목적을 가진 최종 버전입니다. 원본과 다른 점은 (1) 최신 패키지 경로를 사용하고, (2) 각 부분이 무슨 일을 하는지 주석으로 설명되어 있으며, (3) `create_retrieval_chain` 대신 우리가 직접 검증한 수동 체인 방식을 사용한다는 것입니다.

In [ ]:
# ------------------------------------------------------------
# 실전 RAG 파이프라인 (실행하려면 OPENAI_API_KEY 환경변수 + 인터넷 연결 필요)
# ------------------------------------------------------------
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 문서 로딩
with open("knowledge_base.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
docs = [Document(page_content=raw_text, metadata={"source": "knowledge_base.txt"})]

# 2. 청킹 (실제 서비스 규모에 맞춘 값)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

# 3. 임베딩 & 4. 벡터스토어 인덱싱
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./db")

# 5. 검색기
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 6. 프롬프트 + LLM
prompt = ChatPromptTemplate.from_template(
    "다음 참고 자료만 바탕으로 답변하세요. 자료에 없으면 모른다고 답하세요.\n\n"
    "[참고 자료]\n{context}\n\n[질문]\n{input}\n\n[답변]"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(found_docs):
    return "\n\n".join(d.page_content for d in found_docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7. 질의
query = "재택근무는 얼마나 할 수 있나요?"
answer = rag_chain.invoke(query)
sources = retriever.invoke(query)

print(f"Answer: {answer}")
print(f"Sources: {[d.metadata for d in sources]}")

> **예상 출력 예시 (참고용 — 실제 실행 결과가 아니라, 어떤 형태로 나오는지 보여드리기 위한 예시입니다):**
> ```
> Answer: 재택근무는 주 2회까지 가능합니다. 전날 오후 6시까지 팀장의 승인을 받아야 하며,
> 수습 기간(입사 후 3개월) 중에는 재택근무를 사용할 수 없습니다.
> Sources: [{'source': 'knowledge_base.txt'}, {'source': 'knowledge_base.txt'}, {'source': 'knowledge_base.txt'}]
> ```
> 실제 문장 표현은 사용하는 모델과 그때그때의 생성 결과에 따라 달라질 수 있지만, **규정에 있는 사실 관계(주 2회, 전날 오후 6시 승인 등)는 검색된 문서에 근거해 일관되게 유지**되는 것이 RAG의 핵심입니다.

## 11. 정리

### 전체 파이프라인 요약

| 단계 | 이름 | 하는 일 | 이 노트북에서 사용한 도구 |
|---|---|---|---|
| 준비 | 문서 로딩 | 파일을 `Document` 객체로 변환 | `langchain_core.documents.Document` |
| 준비 | 청킹 | 긴 문서를 작은 조각으로 분할 | `RecursiveCharacterTextSplitter` |
| 준비 | 임베딩 | 텍스트를 의미 기반 벡터로 변환 | 연습: TF-IDF / 실전: `HuggingFaceEmbeddings` |
| 준비 | 인덱싱 | 벡터를 빠르게 검색 가능한 형태로 저장 | `Chroma` |
| 질의 시 | 검색 | 질문과 가장 비슷한 청크 top-k개 찾기 | `retriever.invoke(query)` |
| 질의 시 | 프롬프트 구성 | 검색 결과 + 질문을 하나의 지시문으로 결합 | `ChatPromptTemplate` |
| 질의 시 | 생성 | LLM이 프롬프트를 보고 답변 작성 | 연습: `FakeListChatModel` / 실전: `ChatOpenAI` |

### 자주 하는 실수와 팁

- **chunk_size를 너무 크게 잡는다**: 문서 전체를 청크 하나로 넣으면 검색 의미가 없어집니다. 반대로 너무 작으면 문맥이 잘려서 답변 품질이 떨어집니다. 몇 가지 값으로 직접 실험해보는 것이 가장 확실합니다.
- **k(검색 개수)를 습관적으로 크게 잡는다**: 관련 없는 내용이 많이 섞여 들어가면 LLM이 오히려 헷갈려서 엉뚱한 답을 만들 수 있습니다.
- **RAG를 쓰면 환각이 100% 사라진다고 오해한다**: RAG는 환각 "가능성을 크게 줄여줄" 뿐입니다. 검색된 문서에 없는 내용을 LLM이 그래도 지어낼 수 있으므로, 프롬프트에 "참고 자료에 없으면 모른다고 답하라"는 지시를 넣는 것이 중요합니다. (이 노트북의 프롬프트에도 이 지시가 들어 있습니다.)
- **출처(source)를 표시하지 않는다**: 실제 서비스에서는 답변과 함께 "어떤 문서에서 이 답을 가져왔는지"를 함께 보여주면, 사용자가 답변을 검증할 수 있어 신뢰도가 크게 올라갑니다.
- **API 키를 코드에 직접 적는다**: 반드시 환경 변수나 `.env` 파일, 비밀 관리 도구를 사용하세요.

### 다음 단계로 시도해볼 것들

- **다른 벡터스토어 사용해보기**: FAISS, Pinecone 등 다른 벡터스토어로 바꿔보고 차이를 비교해보세요.
- **평가(Evaluation) 해보기**: "질문-정답" 쌍을 여러 개 만들어두고, 우리 RAG 시스템이 얼마나 정확하게 답하는지 점수를 매겨보세요.
- **하이브리드 검색**: 이 노트북에서 본 것처럼 키워드 기반 검색(TF-IDF 계열)과 의미 기반 검색(딥러닝 임베딩)을 함께 사용하면 서로의 단점을 보완할 수 있습니다.
- **에이전트형(Agentic) RAG**: 이 노트북은 "검색 후 바로 답변"하는 가장 단순한 구조(2-Step RAG)입니다. 최신 LangChain은 `langchain.agents.create_agent`로 LLM이 스스로 "지금 검색이 더 필요한가?"를 판단하며 여러 번 검색하는 에이전트형 RAG도 지원합니다. 기본 구조에 익숙해진 뒤 다음 학습 주제로 살펴보면 좋습니다.

이것으로 LangChain을 이용한 RAG 시스템 구현 실습을 마칩니다. 다음 실습 코드에서는 이 기본 구조를 확장한 예제들을 다룹니다.